In [1]:
from sqlalchemy import create_engine, Column, Integer, String, Float, DateTime, MetaData, Table
from sqlalchemy.orm import sessionmaker, declarative_base
import os
import json

base = declarative_base()

In [2]:
class SoilData(base):
    __tablename__ = 'soil_data'

    id = Column(Integer, primary_key=True)
    location = Column(String)
    property = Column(String)
    value = Column(Float)
    unity = Column(String)

In [3]:
def extract_data_from_json(json_file):
    data = {}
    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
        print(f"Loaded data from {json_file}: {data}")

    properties = data.get('property')
    soil_data = {}
    for property_name, entries in properties.items():
        for entry in entries:
            depth = entry.get('depth', {}).get('value')
            if depth != "0-20":
                continue
            value = entry.get('value').get('value')

            if not isinstance(value, (int, float)):
                print(f"Skipping entry for property '{property_name}' due to non-numeric value: {value}")
                continue
            soil_data[property_name] = {
                "value": value,
                "unit": entry.get('value').get('unit')
            }

    return soil_data

In [4]:
soil_data = extract_data_from_json('../data/soil/properties.json') #data\soil\properties.json

Loaded data from ../data/soil/properties.json: {'property': {'aluminium_extractable': [{'value': {'unit': 'ppm', 'type': 'float', 'value': 89}, 'depth': {'value': '0-20', 'unit': 'cm'}, 'uncertainty': [{'confidence_interval': '50%', 'lower_bound': 41.9, 'upper_bound': 188}, {'confidence_interval': '68%', 'lower_bound': 29.1, 'upper_bound': 267.8}, {'confidence_interval': '90%', 'lower_bound': 13.7, 'upper_bound': 548.7}]}, {'value': {'unit': 'ppm', 'type': 'float', 'value': 89}, 'depth': {'value': '20-50', 'unit': 'cm'}, 'uncertainty': [{'confidence_interval': '50%', 'lower_bound': 48.1, 'upper_bound': 164.2}, {'confidence_interval': '68%', 'lower_bound': 35.8, 'upper_bound': 219.3}, {'confidence_interval': '90%', 'lower_bound': 19.5, 'upper_bound': 394.6}]}], 'ph': [{'value': {'unit': None, 'type': 'float', 'value': 6.5}, 'depth': {'value': '0-20', 'unit': 'cm'}, 'uncertainty': [{'confidence_interval': '50%', 'lower_bound': 6.4, 'upper_bound': 6.6}, {'confidence_interval': '68%', 'low

In [7]:
os.makedirs('data/databases', exist_ok=True)
engine = create_engine('sqlite:///data/databases/soil_data.db')
base.metadata.create_all(engine)

session = sessionmaker(bind=engine)()

LOCATION = "Associacao 1º de Maio"
session.query(SoilData).filter_by(location=LOCATION).delete()

0

In [12]:
for name, info in soil_data.items():
    soil_entry = SoilData(
        location=LOCATION,
        property=name,
        value=info['value'],
        unity=info['unit']
    )
    session.add(soil_entry)

session.commit()
print(f"Inserted {session.query(SoilData).filter_by(location=LOCATION).count()} entries for location '{LOCATION}'.")
session.close()

Inserted 36 entries for location 'Associacao 1º de Maio'.


In [13]:
import pandas as pd
df = pd.read_sql_table('soil_data', engine)
df

,id,location,property,value,unity
0,1,Associacao Sombra das Enxadas,aluminium_extractable,89.0,ppm
1,2,Associacao Sombra das Enxadas,ph,6.5,NaN
2,3,Associacao Sombra das Enxadas,nitrogen_total,1.0,g/kg
3,4,Associacao Sombra das Enxadas,phosphorous_extractable,11.2,ppm
4,5,Associacao Sombra das Enxadas,potassium_extractable,108.9,ppm
5,6,Associacao Sombra das Enxadas,magnesium_extractable,147.4,ppm
6,7,Associacao Sombra das Enxadas,calcium_extractable,600.8,ppm
7,8,Associacao Sombra das Enxadas,iron_extractable,59.3,ppm
8,9,Associacao Sombra das Enxadas,zinc_extractable,3.1,ppm
9,10,Associacao Sombra das Enxadas,sulphur_extractable,4.5,ppm
